## Театр LLM

В этом ноутбуке мы пытаемся заставить несколько языковых моделей беседовать друг с другом.

В качестве базовой библиотеки будем использовать OpenAI SDK, который подключается к Yandex AI Studio (Responses API).


In [ ]:
%pip install openai yandex-speechkit


Для доступа к генеративным моделям, потребуются ключи доступа. Разместите их в секретах Datasphere:

In [ ]:
import os

folder_id = os.environ['folder_id']
api_key = os.environ['api_key']
print(f"Using folder {folder_id}")

Создаём функцию для вызова языковой модели:

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id,
)
model = f"gpt://{folder_id}/yandexgpt-5.1"

def GPT(messages,
        system_message=None):
    if isinstance(messages,str):
        messages = [{ "role" : "user", "content" : messages }]
        if system_message is not None:
            messages.insert(0, { "role" : "system", "content" : system_message })
    response = client.responses.create(model=model, input=messages)
    return response.output_text

GPT("Привет! Расскажи анекдот.")


Для поддержки диалога можно использовать **ассистентов**. В Responses API диалог продолжается с помощью `previous_response_id`: ответ предыдущего вызова передаётся в следующий, а история хранится на сервере.


In [ ]:
instruction = "Ты - учитель геометрии в школе. Тебя зовут мисс Радиус."

response = client.responses.create(
    model=model,
    instructions=instruction,
    input="Привет! Что такое число пи?",
)
print(response.output_text)


Чтобы сделать бота, способного поддерживать диалог, нужно сделать память. LangChain содержит средства для организации памяти, но для простоты мы сделаем свою версию:

In [ ]:
response = client.responses.create(
    model=model,
    previous_response_id=response.id,
    input="А если округлить до целого?",
)
print(response.output_text)


Никаких ресурсов создавать не нужно, а значит, и удалять нечего: диалог продолжается через `previous_response_id`.


In [ ]:
# Удалять ресурсы не требуется - состояния ассистента нет, диалог живёт на сервере.


Для упрощения работы создадим класс:

In [ ]:
class Assistant:
    def __init__(self,system_message):
        self.system_message = system_message
        self.previous_response_id = None
        self.messages = [{"role": "system", "content": system_message}]

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        response = client.responses.create(
            model=model,
            instructions=self.system_message,
            previous_response_id=self.previous_response_id,
            input=message,
        )
        self.previous_response_id = response.id
        self.messages.append({"role": "assistant", "content": response.output_text})
        return response.output_text

    def history(self):
        return self.messages

    def done(self):
        self.previous_response_id = None
        self.messages = []

bot = Assistant("Ты школьный учитель геометрии. Тебя зовут Мисс Радиус.")
print(bot("Привет, меня зовут Вася! Я хочу изучить математику! Чему равно число Пи?"))


In [ ]:
print(bot("А если округлить его до целого?"))

In [ ]:
bot.done()

Попробуем сделать диалог двух языковых моделей между собой:

In [ ]:
import time

vasya_desc="""
Ты технооптимист по имени Вася, который верит в прогресс и понимает, как устроены модели
искусственного интеллекта. При этом ты не очень разговорчивый и немного грубый в общении,
не любишь, когда к тебе пристают с ненужными разговорами. 
Отвечай простыми фразами в разговорном стиле.
"""

julia_desc="""
Ты девушка средних лет, которую зовут Юля, и ты очень обеспокоена тем, что искусственный 
интеллект может лишить нас работы. Ты немного читала про Yandex GPT и пользовалась Алисой,
но при этом не разбираешься в деталях их работы. Тебе бы хотелось узнать больше, чтобы 
перестать волноваться. Ты говоришь вежливо, продумывая свои фразы. Общайся в разговорном стиле.
"""

vasya = Assistant(vasya_desc)
julia = Assistant(julia_desc)

msg = "Молодой человек, здравствуйте! Я вижу, вы разбираетесь в технике. Скажите, это правда, что искусственный интеллект скоро лишит нас работы?"

for i in range(10):
    print(f"Юля: {msg}")
    msg = vasya(msg)
    print(f"Вася: {msg}")
    msg = julia(msg)

Озвучим диалог с помощью Yandex Speechkit:

Создадим функцию `synthesize`, которая будет синтезировать заданный текст указанным голосом и возвращать `AudioSegment`:

In [ ]:
from speechkit import model_repository, configure_credentials, creds

# Аутентификация через API-ключ.
configure_credentials(
   yandex_credentials=creds.YandexCredentials(api_key=api_key)
)

def synthesize(text,voice='jane'):
   model = model_repository.synthesis_model()

   # Задайте настройки синтеза.
   model.voice = voice

   # Синтез речи и создание аудио с результатом.
   result = model.synthesize(text, raw_format=False)
   return result

res = synthesize('Привет, как ты?')
res

Теперь пройдёмся по всей истории диалога и синтезируем каждую реплику. Голос будем выбирать в зависимости от персонажа.

In [ ]:
from tqdm.auto import tqdm
res = None
for msg in tqdm(vasya.history()[::-1][1:]):
  x = synthesize(msg['content'],'julia' if msg['role']=='user' else 'zahar')
  if res:
    res += x
  else:
    res = x


Послушаем результат прямо в Jupyter Notebook:

In [ ]:
res

Используем следующий код для записи результа на диск:

In [ ]:
res.export('LSH_dialogue.mp3')

В заключении очистим локальную память ассистентов:


In [ ]:
vasya.done()
julia.done()

## Yandex ART и многоагентное рисование

Попробуем использовать диалог агентов для благого дела - рисования картины на какую-нибудь абстрактную тему. Для начала научимся вызывать генеративную модель для рисования - Alice AI ART:


In [ ]:
from PIL import Image
from io import BytesIO
import base64

art_model = f"art://{folder_id}/aliceai-image-art-3.0"

def generate(prompt):
    res = client.images.generate(model=art_model, prompt=prompt, size='1536x1024')
    image_bytes = base64.b64decode(res.data[0].b64_json)
    return Image.open(BytesIO(image_bytes))

generate('бедность')


Теперь создадим двух агентов, как в предыдущем примере. Попросим их придумать, что можно изобразить на картине.

In [ ]:
import time

topic = "бедность"

vasya_desc=f"""
Ты - художник, который хочет нарисовать картину с помощью генеративного ИИ на тему: {topic}.
Ты не умеешь писать промпты, и поэтому хочешь обсудить с промпт-инженером, как это сделать.
Ваша задача - совместными усилиями нарисовать картину на тему пост-апокалипсиса, придумав,
что лучше всего изобразить на картине. Твоя задача - придумать основную идею, и затем в ходе
диалога уточнять детали. Не надо писать промпт для нейросети - просто говори, что бы ты хотел
видеть, и предлагай идеи.
"""

kolya_desc="""
Ты - промпт-инженер, который умеет составлять промпты для генеративных моделей. Твоя задача - помочь
художнику нарисовать картину. Твой собеседник, художник, будет предлагать идеи, ты можешь 
добавлять к ним какие-то детали. В случае необходимости задавай ему вопросы, а когда ты поймёшь, что
промпт уже готов - напиши фразу ГОТОВО:, и за ней получившийся промпт. Не пиши промпт и фразу "ГОТОВО", 
если ты не выяснишь все детали у художника. Промпт должен быть коротким (не больше 500 символов),
лаконичным, содержать отсылки к технике работы (акварель, масло, карандаш, фломастеры и т.д), и 
возможно к художественным стилям и приёмам.
"""

vasya = Assistant(vasya_desc)
kolya = Assistant(kolya_desc)

msg = f"Добрый день! Я хочу нарисовать картину на тему {topic}. Вы поможете мне составить промпт?"

while True:
    print(f"Вася: {msg}")
    msg = kolya(msg)
    print(f"Коля: {msg}")
    if "ГОТОВО" in msg.upper():
        break
    msg = vasya(msg)
    if "ГОТОВО" in msg.upper():
        break


In [ ]:
prompt = msg.split('ГОТОВО:')[1].strip()
print(prompt)

In [ ]:
generate(prompt)

In [ ]:
vasya.done()
kolya.done()